# Import

In [ ]:
import numpy as np
import pandas as pd
import torch

import alloc

# Data Reading and Pairing

In [8]:
TASK_SEEDS = {
    "sst2": [0, 1, 2],
    "mrpc": [0, 1, 2, 3, 4],
    "rte":  [0, 1, 2, 3, 4],
}

pairs = []
for task, seeds in TASK_SEEDS.items():
    for seed in seeds:
        g9_path = f"../cache/g9A_{task}_s{seed}.pt"
        g3_path = f"../cache_g3/g3A_{task}_False_s{seed}.pt"

        g9_dict = torch.load(g9_path, map_location="cpu")
        g3_dict = torch.load(g3_path, map_location="cpu")
        pairs.append((task, seed, g9_dict, g3_dict))

print(f"Successfully loaded and paired {len(pairs)} sets of weight files.")

Successfully loaded and paired 13 sets of weight files.


# Overlap Calculation

In [6]:
results = []

for task, seed, g9_dict, g3_dict in pairs:
    layer_overlaps = []

    # traverse 72 layers
    for name in g9_dict.keys():
        a_cross = g9_dict[name]
        a_self = g3_dict[name]
        overlap = alloc.subspace_overlap(a_cross, a_self)
        layer_overlaps.append(overlap)

    avg_overlap = float(np.mean(layer_overlaps))
    print(f"[{task.upper()}] seed {seed}: mean overlap = {avg_overlap:.4f}")
    
    results.append({
        "task": task,
        "seed": seed,
        "overlap": avg_overlap
    })

print("The overlap calculation is all done.")

df = pd.DataFrame(results)
df.to_csv("../results/h4_overlap.csv", index=False)

[SST2] seed 0: mean overlap = 0.5069
[SST2] seed 1: mean overlap = 0.4809
[SST2] seed 2: mean overlap = 0.4898
[MRPC] seed 0: mean overlap = 0.6626
[MRPC] seed 1: mean overlap = 0.6674
[MRPC] seed 2: mean overlap = 0.6832
[MRPC] seed 3: mean overlap = 0.6549
[MRPC] seed 4: mean overlap = 0.6440
[RTE] seed 0: mean overlap = 0.6626
[RTE] seed 1: mean overlap = 0.6674
[RTE] seed 2: mean overlap = 0.6832
[RTE] seed 3: mean overlap = 0.6549
[RTE] seed 4: mean overlap = 0.6440
The overlap calculation is all done.
